## 1. Construct Model Input Features

### 1.0 Import the dataset. Clean data, discard trades in Pre-market Trading and After-Hours Trading

In [39]:
import pandas as pd
import glob
import os
import re

file_pattern = 'dbeq-basic-*.trades.csv' 
file_paths = glob.glob(file_pattern)

dataframes = {}

for file_path in file_paths:
    match = re.search(r'\d{8}', file_path)
    date_part = match.group(0) if match else 'unknown_date'
    file_key = f"NVDA-{date_part}"
    df = pd.read_csv(file_path)
    
    df_cleaned = df[['ts_event', 'price', 'size']].copy()
    df_cleaned['ts_event'] = pd.to_datetime(df_cleaned['ts_event']).dt.tz_localize(None)  

    trading_start = df_cleaned['ts_event'].dt.time >= pd.to_datetime("13:30:00").time()
    trading_end = df_cleaned['ts_event'].dt.time <= pd.to_datetime("19:59:59").time()
    df_filtered = df_cleaned[trading_start & trading_end]
    
    df_filtered.loc[:, 'ts_event'] = df_filtered['ts_event'].dt.strftime('%Y-%m-%d %H:%M:%S')

    dataframes[file_key] = df_filtered

# Display the head of each dataframe with the index reset for display purposes only
for key, df in dataframes.items():
    print(f"\nHead of {key}:")
    print(df.reset_index(drop=True).head(5))



Head of NVDA-20240828:
             ts_event   price  size
0 2024-08-28 13:30:00  128.12   200
1 2024-08-28 13:30:00  128.09    67
2 2024-08-28 13:30:00  128.09    33
3 2024-08-28 13:30:00  128.09    67
4 2024-08-28 13:30:00  128.09    33

Head of NVDA-20241004:
             ts_event   price  size
0 2024-10-04 13:30:00  124.93    43
1 2024-10-04 13:30:00  124.93    39
2 2024-10-04 13:30:00  124.93     5
3 2024-10-04 13:30:00  124.93    13
4 2024-10-04 13:30:00  124.92    11

Head of NVDA-20240730:
             ts_event    price  size
0 2024-07-30 13:30:00  111.515    90
1 2024-07-30 13:30:00  111.520    10
2 2024-07-30 13:30:00  111.520    18
3 2024-07-30 13:30:00  111.520   100
4 2024-07-30 13:30:00  111.520   100

Head of NVDA-20240613:
             ts_event    price  size
0 2024-06-13 13:30:00  129.390   100
1 2024-06-13 13:30:00  129.400   100
2 2024-06-13 13:30:00  129.400    74
3 2024-06-13 13:30:00  129.400    26
4 2024-06-13 13:30:00  129.445     4

Head of NVDA-20240624:
    

### 1.1 Arrival Price Matrix

In [40]:
import numpy as np

arrival_prices = [df['price'].iloc[0] for df in dataframes.values()]
arrival_price = np.array([arrival_prices])

print("Arrival Price Matrix:")
print(arrival_price)


Arrival Price Matrix:
[[128.12  124.93  111.515 129.39  123.3   118.2   117.01  138.75  134.135
  106.42  108.07  134.69  142.92  137.83  128.42  116.04  128.43  131.91
  119.17  137.73  127.04  121.81  121.11  113.64  103.85  122.02  116.88
  117.52  120.32  104.85  137.575 127.44  126.12  121.77  139.35  122.9
  115.94  127.49  125.03  105.38  105.61  116.18  140.91  135.91  141.73
  130.04  118.76  143.03  107.95  123.93  129.95  120.93  121.92  127.33
  134.05  128.235 113.07  137.43  139.79  132.93  121.63  119.52  107.87
  116.5   119.06  148.62  112.88  123.1   121.2   121.37  116.77  103.84
  112.45  118.31  130.25  124.58  142.02  121.37  136.46  140.86  134.03
  145.85  102.    131.2   125.86  118.51   92.02  109.37  126.79  140.285
  124.3   116.41  123.47  121.86  116.47  139.54  124.99  124.11  120.45
  117.3   133.955 130.72  120.39  138.14  129.56  105.03  130.36 ]]


### 1.2 Terminal Price Matrix

In [41]:
terminal_prices = [df['price'].iloc[-1] for df in dataframes.values()]
terminal_price = np.array([terminal_prices])

print("Arrival Price Matrix:")
print(terminal_price)

Arrival Price Matrix:
[[125.6    124.84   103.68   129.51   118.13   115.62   116.1    137.98
  132.63   109.05   102.92   135.37   143.56   131.66   126.42   107.94
  127.23   134.84   114.26   137.78   126.93   116.94   122.64   111.58
  104.03   123.55   119.22   109.48   118.07   106.45   132.5    125.81
  126.37   120.91   136.91   122.61   113.43   128.1    128.27   106.21
  104.73   113.1    141.53   127.44   146.41   123.85   122.9    140.64
  108.01   121.38   131.94   122.82   124.52   128.46   134.81   129.13
  112.13   139.43   130.75   131.04   127.7    119.5     98.96   120.89
  119.05   147.025  117.06   125.28   126.1    117.6    116.74   107.28
  116.12   121.34   132.875  123.52   139.58   118.04   138.06   140.32
  134.94   148.47   104.99   135.68   129.21   117.97   100.64   116.82
  123.975  141.36   129.96   118.85   124.28   121.33   116.28   139.34
  127.705  124.06   121.87   117.83   135.7    128.2846 123.5    143.67
  126.44   107.15   131.4   ]]


### 1.3 VWAP Matrix

In [42]:
vwap_values = [
    (df['price'] * df['size']).sum() / df['size'].sum() 
    for df in dataframes.values()
]

vwap_matrix = np.array([vwap_values])

print("VWAP Matrix:")
print(vwap_matrix)


VWAP Matrix:
[[125.6853417  124.15813078 105.67133688 128.70494895 119.98125112
  116.24768086 116.37068179 138.02238042 132.6282652  109.17811989
  102.90877578 136.04380387 143.3230543  132.34301292 126.14452227
  110.02515149 127.81193013 134.04667414 116.39545114 137.78011575
  126.76947719 117.80523929 122.32143274 113.06105931 105.34947355
  123.59185895 119.1054829  111.80607047 119.18376838 105.47287148
  133.68502231 127.14730714 124.58574212 120.80332331 139.05126714
  123.25504563 115.01016942 128.74278827 127.80067967 106.97585691
  104.89126232 113.80647969 142.53536296 129.09019916 144.43284527
  126.23033985 121.67311581 141.17104184 107.12647418 120.90250747
  131.68004467 122.78342769 123.89581984 128.10427712 134.88300937
  129.94060241 112.61293423 139.36821313 135.34455832 131.72652218
  125.48651448 119.01332317 103.18093844 119.64702785 119.10167092
  147.64774761 114.74130565 125.18865797 123.86744614 120.62159028
  116.46261752 105.26726131 114.7983466  120.1700

### 1.4 Total Shares Matrix(after indicating buy OR sell)

In [43]:
total_shares = []

for df in dataframes.values():
    shares = 0  
    for i in range(len(df) - 1):
        if df['price'].iloc[i + 1] > df['price'].iloc[i]:
            shares += df['size'].iloc[i]  # Buy action, add size
        else:
            shares -= df['size'].iloc[i]  # Sell action, subtract size

    # For the last row, assume it is a buy action
    shares += df['size'].iloc[-1]
    
    total_shares.append(shares)

total_shares_matrix = np.array([total_shares])

print("Total Shares Matrix:")
print(total_shares_matrix)


Total Shares Matrix:
[[ -1340979  -1140093  -3320546   -591647    397491  -1532693  -2556795
    -472472  -1942132   -791707  -5635943  -1294562  -1967622  -1869349
    -647313  -2634709  -1018534  -1502372  -6975732   -847866 -10679816
   -2058738     48838   -950108  -5847932  -1111602   -149454  -2126114
     209841   -689167  -1354344   -658847   7306768   -746392    558201
    -346093   -627191  -1545502  -1727634  -2684901   -460013   6208342
   -1432135    870914   -900582  -1520408   -975190    190347    366709
   -2522765   3296372  -1260889    990694   -741139  -1055310   -371949
   -2398848   -297588  -3283437  -1582544  -2094662   -692090  -1307604
   -2021837  -1200964   -785082  -5189456    166482   -463393  -2498250
    -986077  -1135253   -936661   -718972  -2491806  -1148277  -1617450
   -1962291  -2159916  -1169986   -724596   3694903  -1528290    872453
   -1071387  -1385441  -5313080  -1842403  -6237059   -594386   -830938
    5068289   -861608   -977428  -1085001  

### 1.5 Imbalance Matrix

In [44]:
imbalance = np.multiply(total_shares_matrix, vwap_matrix)
print(imbalance)

[[-1.68541404e+08 -1.41551816e+08 -3.50886535e+08 -7.61478969e+07
   4.76914675e+07 -1.78172007e+08 -2.97535977e+08 -6.52117101e+07
  -2.57581598e+08 -8.64370818e+07 -5.79987994e+08 -1.76117139e+08
  -2.82005595e+08 -2.47395279e+08 -8.16549891e+07 -2.89884257e+08
  -1.30180796e+08 -2.01387970e+08 -8.11943473e+08 -1.16819076e+08
  -1.35387469e+09 -2.42530123e+08  5.97393413e+06 -1.07420217e+08
  -6.16076558e+08 -1.37384958e+08 -1.78007908e+07 -2.37712452e+08
   2.50096411e+07 -7.26884224e+07 -1.81055508e+08 -8.37706219e+07
   9.10319114e+08 -9.01666341e+07  7.76185564e+07 -4.26577085e+07
  -7.21333432e+07 -1.98972237e+08 -2.20792799e+08 -2.87219585e+08
  -4.82513443e+07  7.06549548e+08 -2.04129882e+08  1.12426462e+08
  -1.30073621e+08 -1.91921619e+08 -1.18654406e+08  2.68714843e+07
   3.92842422e+07 -3.05008614e+08  4.34066412e+08 -1.54816273e+08
   1.22742845e+08 -9.49430758e+07 -1.42343389e+08 -4.83312771e+07
  -2.70141312e+08 -4.14743078e+07 -4.44395331e+08 -2.08463017e+08
  -2.62851

### 1.6 Total Daily Value Matrix

In [45]:
total_daily_values = [
    (df['size'] * df['price']).sum() 
    for df in dataframes.values()
]

total_daily_value = np.array([total_daily_values])

print("Total Daily Value Matrix:")
print(total_daily_value)


Total Daily Value Matrix:
[[5.97953669e+08 4.58515356e+08 8.96035452e+08 3.01132385e+08
  9.62801428e+08 5.01251049e+08 1.11703042e+09 4.13602834e+08
  8.28232216e+08 3.12816385e+08 1.13654479e+09 6.08946487e+08
  6.06001118e+08 7.53680357e+08 2.47866796e+08 7.05374877e+08
  4.30789344e+08 5.39592554e+08 1.06739401e+09 3.56005908e+08
  1.95666989e+09 5.24170878e+08 3.91332685e+08 2.85545655e+08
  8.98005655e+08 5.43858560e+08 6.81120188e+08 8.44135161e+08
  9.66497529e+08 5.15502140e+08 4.33974469e+08 2.66196238e+08
  1.66687077e+09 2.55999638e+08 1.06917840e+09 1.63401309e+08
  9.48127160e+08 3.81163366e+08 5.76581201e+08 7.22151541e+08
  3.37394808e+08 1.21386151e+09 4.78798662e+08 6.15309983e+08
  4.50178402e+08 4.43584775e+08 3.85837618e+08 4.99494345e+08
  6.19803677e+08 6.34670338e+08 7.41456885e+08 4.61985539e+08
  8.74177931e+08 4.81565627e+08 3.26033275e+08 3.44620279e+08
  5.83005719e+08 2.27582996e+08 8.47214031e+08 4.01094087e+08
  4.40056109e+08 4.54169123e+08 6.05450476e+

### 1.7 Calculating $\sigma$

In [46]:
import numpy as np

minute_returns = []

for df in dataframes.values():
    df_resampled = df.set_index('ts_event').resample('1T').last()
    returns = df_resampled['price'].pct_change().dropna()
    minute_returns.extend(returns)

minute_returns = np.array(minute_returns)

sigma = np.std(minute_returns) * np.sqrt(390)

print("Sigma:", sigma)


/var/folders/0_/58fmvk7j6w99vs8qqzxlk3hh0000gn/T/ipykernel_10688/3093402748.py:7: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Call ffill before calling pct_change to retain current behavior and silence this warning.
  returns = df_resampled['price'].pct_change().dropna()


Sigma: 0.028277032792718145


## 2. Build Rregression Model for estimating $\eta$ and $\beta$

In [48]:
import numpy as np
from scipy.optimize import curve_fit

H = vwap_matrix + (arrival_price / 2) - (terminal_price / 2)
X = imbalance.flatten()
V = np.mean(total_daily_value[:, -10:], axis=1) 
V_broadcasted = np.tile(V, (1, H.shape[1])) 
V_flatten = V_broadcasted.flatten()

sigma_broadcasted = np.full_like(H, sigma)
sigma_flatten = sigma_broadcasted.flatten()

H_flatten = H.flatten()
XV_combined = np.vstack((X, V_flatten, sigma_flatten))

def market_impact_model(XV, eta, beta):
    X, V, sigma = XV
    return eta * sigma * (np.abs(X) / V) ** beta

initial_guesses = [1, 0.25]

popt, pcov = curve_fit(market_impact_model, XV_combined, H_flatten, p0=initial_guesses)

eta_estimated, beta_estimated = popt

eta_estimated, beta_estimated


(4362.6073024505195, -0.004502353149282298)

In [50]:
from scipy.optimize import curve_fit

# Define the updated generate_data function with eta and beta as inputs
def generate_data(X, V, sigma, eta, beta):
    """Generate new 'H' data using the market impact model."""
    return eta * sigma * (np.abs(X) / V) ** beta

H_generated = generate_data(X, V_flatten, sigma_flatten, eta_estimated, beta_estimated)

def market_impact_model(XV, eta, beta):
    X, V, sigma = XV
    return eta * sigma * (np.abs(X) / V) ** beta

popt_generated, pcov_generated = curve_fit(market_impact_model, XV_combined, H_generated, p0=initial_guesses)

eta_estimated_generated, beta_estimated_generated = popt_generated

bootstrap_eta = []
bootstrap_beta = []
n_bootstraps = 1000

for _ in range(n_bootstraps):
    noise = np.random.normal(0, 0.01, size=H_generated.shape) 
    H_bootstrap = H_generated + noise

    try:
        # Perform curve fitting on the bootstrap sample
        popt_bootstrap, _ = curve_fit(market_impact_model, XV_combined, H_bootstrap, p0=initial_guesses)
        bootstrap_eta.append(popt_bootstrap[0])
        bootstrap_beta.append(popt_bootstrap[1])
    except RuntimeError:
        continue

eta_std = np.std(bootstrap_eta)
beta_std = np.std(bootstrap_beta)

print("Eta Standard Deviation:", eta_std)
print("Beta Standard Deviation:", beta_std)


Eta Standard Deviation: 0.06090374999701027
Beta Standard Deviation: 8.872434217068625e-06
